# Resume additional P1 diagnostics after MemoryError

Upload **this single notebook** into `~/P1` and use the existing native P1 kernel. It recovers the causal/robustness studies in `additional-diagnostics-20261024-v1`. Keep the original package, results, checkpoints and partial raw files.

This is a separate recovery from `P1_Resume_After_MemoryError.ipynb`, which handles the first timing/application study. This adapter supports the additional runner's ordered `map` interface and stream checkpoint format. No experiment source, seed, sample size or comparison is changed.

One fresh spawned process executes each original stream, with at most eight such workers alive. Submission is bounded. Before resuming full studies, the selected smokes repeat in a separate validation folder and must exactly match saved original compact records; their raw traces are replayed through the unchanged auditor.

Six local host-process engineering tests pass. **Native recovery validation remains pending on the server.** The allocation failure is established; the precise memory-retention cause is not.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.home() / "P1"
RUN_ROOT = PROJECT_DIR / "additional-diagnostics-20261024-v1"  # Keep the original run
PACKAGE = RUN_ROOT / "P1_Additional_Studies"
RESULTS = RUN_ROOT / "results"
WORKERS = 8
FAMILY = "all"             # "all", "causal", or "robustness"
MODE = "full"              # "check", "smoke", or "full"
PYTHON_OVERRIDE = None     # Prefers the interpreter recorded by the original notebook
QOALA_REPO = None          # Optional existing pinned Qoala repository

assert FAMILY in ("all","causal","robustness")
assert MODE in ("check","smoke","full") and 1 <= WORKERS <= 8
assert (PACKAGE / "source/study_runner.py").is_file(), "Original additional package missing: check RUN_ROOT; do not delete the run."
assert RESULTS.is_dir(), "Original results directory missing."
LOGDIR = RUN_ROOT / "additional-memory-recovery-v1" / FAMILY
LOGDIR.mkdir(parents=True,exist_ok=True)
print({"run":str(RUN_ROOT),"family":FAMILY,"mode":MODE,"workers":WORKERS})


In [ ]:
import base64, hashlib, io, json, os, signal, subprocess, zipfile
PAYLOAD_SHA256 = '5302b86a23bad6068556fce95aa6a0b93b7c5a80b290267b0e4990f232796604'
PAYLOAD = 'UEsDBBQAAAAIAAmdWF30hL9RTAIAAC0IAAAcAAAAYWRhcHRlci9MT0NBTF9WQUxJREFUSU9OLnR4dK2UTW/bMAyG7/4VQk4b0Br68OeuOw5rgW53QZWZRIgteZKcJij230c5wYAOLWID00W2RPJ5KVJ63cARbNx8IZuth7CXL84fwEs4mbi5I5vRdLjXpC9QB+lDkAfzjEucFZzi8ovqexlAO9sFXG5zUQuKg7Gq5bSs4J5Wv7PXRRhWvMPhNX2XUzVl4tCWNqUoxCqOWMyp8pa3aTSiKug6Srsim0KIGcOamnK+isP5imwaNmNKwTkv67pcBSoXg5ocAak6BWtrVtTNGo6gizk0p7QWDW3ZTBO0LJrFGLYGw2hVlnVxwbStmLMJ0YMakgH6wAn0FI2zySPFfuP/xpwtMl+URLXurFiL1ShSElw0RflGFV+gKkKIclCjHGBw/gzeOy9H70a1U7hHPkncNVbK/Akdj+DPX/egDyG/4fiZ5HlO3OEKMCEYu5NWRXMENINtb3b7KLWy1kXZq8nqvQxR7W4z18R6K8P5Djx0s2wT5LObbIe/ynZyChBkZ0I0VseUh4YQbmpZH/BfQWZnrOrlpUx4gtgZOKUY2zg3h4Yxle+mkKWBPhBwaZrkNQ2A0yxfJ8LojI23D2JhnA/w17uAxQzgj+iCtthS0xgxg1H5aFS/VMKSWH9lZPf/ZWRPypKKJCGBGIt3q2IiZNnjtyz7YU5k70K8vzYBAYtCATx28cUjJw9zM5Nfk7JxGoi/5kceHn+SS0mhI73T+AKcc/Ld6cNs47tAlAeMMozOK3TozRb0WfdAtuYUJzyDO4K3AoOMCBzwASJuitoNEPLsD1BLAwQUAAAACADynFhdemlyfmkJAAD7EwAAEgAAAGFkYXB0ZXIvUkVBRE1FLnR4dHVY224jNxJ951fwbbKALmNnsghizIMy0iyE8Q2SnEX2xaC6KYlwd7NDsi33POy376ki2WonWWDgsVvsYl1OnTqlzerLw2+rjVwsl+vd+uF+cSsfr+RyvfjX/cN2t/6ylYuvO3x+t7p72Py+2mweNkI8eY1Tzxvtu1o/L8rSBGMbVT0vjTo21gdT+Jlp+2YvD9bJcNLyoEylS1mozqtq7uy+86HR3kvXNTOxDtJ4WZrDQTvdBHlwth7fcAjaPd/p2rp+5Zx10fhEnk+mOEmnC/uqnY8XGeeDCKY2zXGu2rYyhSLvpA9d2c/krzYMb/SysUHvrX3x8KwhX6Shf6VuNX7AE68dDgpVVTba8TO5Urj0RevWSxO8tOdGWmeOBhlgE87aIFVTStuFtguUg1oFPxPiYc/2Sk5H5/Qv7LIaEjj1nQlanq170U7qtxPSFXC85tAp3ErLfWeqEtEJJRu49KrlH51qQlfL1tkCObVuQnYbWdiuKilGmfzX2VKwyau5VzCA00K/FbrlTKFkRWVR4+N306YgZnJ3Qom0D2pfGX/SXl5yksOZxLtEi/wavF9p9SK97Vyh6X0NB01D8fjgtKrl9c/XVHdFV5aofInC+K4KNxJO6nTKi1r18kRuIr6yKwhGJ128tBbGvNzrAx1OsU9bayvpT10oUReknK4dqqPfdNEFxOd05xFCyl/MNxwpnAUka9X0+fIc9xsQCAtClarFr8AICqMoEuUIAw2QB+dP0rfq3MDF5I9scXi4PxoFbg1AqIKo0Svy3w+bb6vNVvoOsEqvcXrh2kw+pMzUqsWJfW28p4zDp73tgNGSaplMCNQKDjWEDkBTR/e1NHVb6RpwjtXi0BePa9zxisKhoJpg/9iHEz79cXb1kdArSksZQkGd/qMzSHGt3p6D8i/+GUE9FwBjKX9ADXJVKgDM/QM5vwe6CKnkn6M+HlHEFm1oYJiSioe6jC4OKYq1mAC75HTQgnJlmmmCTKsMMHScoDFDwC+eftMl/afoBenNd+3j68oZz+CsTYUrJyJCcY4cB1sAJgwjOjzACTgjbE+kU2ep4CmBhVpZwbfeE1iRCEZPOWNsFSfVHBF8hBalN0MEpgZ+iDcn62wQTrzqRjUFmSu0acOM8hZdQi/j3sFPACs4U4RU+NJ4qhkyhxd836B7wbiyVEHRx+Qb6imIWkYAP1T2fMN842u0rtQNEq41ZVMGXMAvooZ7TUQN8LTWKVDFwbwF9DbR13L1dfF0u8NMuF/K7WpDg+NpuxLiaiaf2sqiH8g+NYPX1WFaWEDOUDdknoVfgOt/55gxlAS6kd7Qb8ZTNZO/QkqaQrp5Nc42BNyZ/Aa+jdbHVEtGNDM58diYGH4hI5Kvmo8ItrxMqOn1x+t/Xn28/jR9vZqL63TF18Xd+vb3zx+Qog8TefewXH3+cOj4j9Rln3+mlksjJE00duQy1Wbix5ncwL9FVUV8DwkANx7yrAJkdW47IkaHDwk5ZNtRa6fpSaH8JeyIlFzhGj1Y6TGq1FERSdGFr8Z2nm+uzPEEoH2KTmWkOh6zfp5MDtXwtX3hZsA4xACjouqC+Zs+GBqL3AN/KHyOnqMpM8zXV7BYmWaErUpqbBirVM8JMI77LDhV0DVEOjCV6Ib8UEWInYz/1dFpJrHInjF/VPMLu7JX4EC8Qnj9aSYhLHgSpgilptE9vEB1TX79iYVGjADawefkGCIyB+IuOD3Pbo1aGpWhSVD1klmaEpiHyCILoPiErFEU8BVp4ySNr0QTAwjO/B9ypEqyjqBre7KFuQqkzJD9YAgiSOqIgNN0HkYl2e7jiEBTf41ji3M3KhcPvaLqGIYkxyhZUxZRKDbrNJrqq9h8cJ7wVwpOaeJqgAh/JBKIiM+sTia/k0LRVTVvLURaP1euhmXAF5zbgwwjsKNjBAk4pyNd1tGeRUAxbYSCJkKC9dpM3kKaVPKETp+mkSrGhJeQXloecIOqGZRUBhHRcIn+KQbltNeQIsg2Mre+X64eV/hxvwMbbreQz1sIDnAyVGfgYmVJMXSGpgYEMaIMI005YSbM1BMp5QNxeKTSOPIJYvnIhWroWORF/HCkFwatQaWK6MbsG1oSsYbOzyt7nP+lOxOOEbA+y/+sH8EulASaeSVzQgHELGPSiIeYJRGIxSA2pSYK6CBxw4TJAJUHrG3bS3u48B2zRj+RidoGbF9IGkOoa9CnBI6Bxob34f+R3EFADBT5qN2UH4rN0/0MOX3J5Y1IQQ0CS6ldXg/k36wHQ35YHSlxWUbeSfrxmpBm+3hduNQU+Ig7Ew/L2/XderfYRYTEjrsovQreEM8EWOBCJ8xcFKjuM1SPHXCEUUFzEwKS4UQtVnSOnU0vIc6QB2OCT8IveXhW3Lvrg2w0+YAxPxEoLl71Ay7RU5vFHZUJBR1m3yf5A54c9BlKL4GFmHVgM5ZDtJzcx9ZBs3O/pb7PCh9RfYmDMzucaEJeXX+Ehqu7KnbGjRjtin8+++mnd2c5opESjyxE6CSFUZBMZFF0Ub0xJ5Px6HovNwdVN/cAGwGPZ2amSAE+AUHeDGFnhq7MAYHXOt0OMgWIu8PBFIbUjBC8Qw50zovM43oZlSFtTZvtljB2a5ruTX4zv2KEDF4XR2e7Fs4HlTSfGBTDMFWALMepJ5zIk9EQc8WpnzKS0fhd6utImhEiDMTZ+5WJhe932nggEJukdmmw8mqT88R+jBCMMA70uqCdoCkAO+do6NMqN6wTnPwbmmFxdYC4JwxPB62taddPipfJR/Ak4kGS8J5XZT5J19YAC3cKZoO3FQmEPQgIgXE26HISuyjBRkPXNunbCeKkqW+xt6JGg+YYMaGi7yAyjqm/SVsgYmInkZX6/DLE50nyJ3FPC0pX11hJkliIugGDSjmgihRFGtOCJjdhHal6N9RYDY7JJs5a0kBTmvhRWGVll8lU+LRwJTK8CGBcSiODANLI1p55zURrgSLrCHDQdtoVwIUFJ+23xe16yUQmxNa8/c0WgZZxvP3n/dReNtjJUMkL9hMLTsTo+x163IL5yYvJeGvnLoxPmXgiwN8vBQ3UWm4Dp+nBmIEyiUyyihpCZQXuupZYaqSkYj0meYWjU0dnQj+JKEfRpvxtSZml+EVny92wRdUYS6lLY98RQxR9UelhwYrfnmQFMuI13gUVfZEAOJasu7I6GqYWLZsR9VBVMWG8BXJ2Sgf0UhLGG5X4H1BLAwQUAAAACAAJnVhdYQWmxMUAAAAaAQAAIwAAAGFkYXB0ZXIvUkVDT1ZFUllfU09VUkNFX0hBU0hFUy5qc29uLc5BasQwDIXh/ZwiZF2KZNmW3MsES7JLYJgMSRgIpXdvhmb/+N7/cxuG8bku1rZtWpsddp8f35/PY/waxpIgcgPTBt1YsrWojaiWGiyV3sCrBNSOOUqWkBpHCrG4CKORpfHjrZ/q8mrrVN3nfV4e9X7xmUUwVCQmDgwBg6Cyi5OjYypWOhgCnxsgUXDNVq0U6JhQGv/ze9v26fo4LvndaBmdxRQIWEvC3Hs8swlTi101sRFDkaydTCl6ilC8YPAIOt5+b39QSwMEFAAAAAgAy5xYXWYa/I4uBQAAhgwAABwAAABhZGFwdGVyL3Byb2Nlc3NfcmVjeWNsaW5nLnB5fVbbbuM2EH3XV3C9D6aysrpB31wI6DZwikWDbpBL2zeBliiLa4lUSSqO8/WdoUhZcrw1ElgWObdzDme4WCzuj7ZWkvycXn9eFartmBXbhpNKc1OvOq0Kbgzhr7zordKkgv9eFjWTO17CJvXGJTkovefapFG0YUVNWibZjmtia81ZSdRBogNW2OZIlOTEdOwgwRqeV4MpuR/i3CvVbHyoNPpqyXclwNjWzJ5S2HJIghPNba+lkDsirIFfpm9sSm6UNPzfnksMBlatMpb8HUmo6oWHRMGXMDYhTJZEKjJbJFBa2XADlhAEIkuXc1H3cp9GT7VAa8u1ZA1Ruqi5sRrsAUFWsg4WSKnAWirIuBQWYhzBoONatJAUMarXBU+jxWIRibZT2pJCyaLXGlbTqoeawJoZUlQRoNviqoWAjdiS035841EOXr4bJYNF0/ACUzLBpERIws4WgBKeWIAvvFYmPEEGLsvw20LqUfSw+XKX3z98u9k8PuabfzY3z0/fHkgGiaYX2IuikldEyBe157QCxWA+CdM7k+wP+BWvIwIfY5m2vMwwRgooVXmheqhP09itW31cD0yT4IVeOTdXV96R21cJIKQ5Dk7x85HcCdm/QjFYAwikz1v2qkHLQpI/xG8pcVzCHyMQd1R6LXb16sCQyJbpfTJxiJwy0nJmgCTHpqpAISAOODYNtxzkuv3JIKwgB1AqI8VOq74jjWiFTUdXnRbSUmQsLfu2M7QUhaX8BVxmS3fw8kGMOQjVLpNOlJky6Y5beKLxKSfnjLN9DnXle7HNAnW4V/cGBELHVw/Pj19+3+SPm7vbOB3hmDs7AIi54SCx0lziZOX5iuOkanpTZ0+65/FANhyT3KNIvUaT/2H+IGxNLoqKQmYeAJNdJ22Xe3eZ/47xgIR+cKJ86AFZWEhNvwXY6SDCy6mkg40X28fQh4ipe1tC4xqpNaHtIN+qR3kMhxXewx5YN32BplXfDER71Q4BoqhoGPi9RXJ9kM2sAEQwz4UUNs+p4U2VTFH4E8CdAuF+jyfgBIGoyPBurZkwnDwdO77RWmm6fJam7/AsQOcde6nqXJ9Yk+Un6GPUuGXq3cbx6BcTSkPwUx4o8rN2gsoLy3TpWv0ynuY3deU2O0nlLYcxBOL+kHkjX8FfrOlDCW4Bj6yGfiY0L5dnGQ4Tx2TQk57c47QlzWQ1eY5H/L1gHPqjXM66zVRujuBp4CC5yVFIpvVeFuEYv2XdeXABSmMwj02CpxGEN1DvxpERbzy7nrM/LnzIri8g6HpeGFQeRUMm3iaIclm6YOjWB0fs3SvegGPXH1oFbVFJUdD4k9914sQ1ngxroG+io6dq4viXDvyDXjI3m+gpLALBJbzDt+tZd8JhgIhlEsU1eI9nO/hrwTtLHq3qvmIsBDHMj1sGSZ+1TpdCyjp8ctAHBuf0Q7reCfa70QdehnKcJxqvQ3SmhHyqsHkZgKe7HoxFbsFkP27BKudNuRZwIfPJzpdcFlmoo1PQqypL43d7DkzYkUzYPScSUqWf0wTer845fe/Kd9kqtM6gSwzxfvdR8KYMTfB8cUTghOj5GA8w411ihADOE5MFb2g8aZ0cJ5TvnfF6cjine15P7XUvZJm84NlI7HbC0Pw8+zlAHYBIfjKEzv1VLUM3iCYyiojG5/1hkF3069m1DROajnoTxK602CEK2Q8uVl5BYVuIfXGS+g7wANMbWBoHAd5Hi9kcYA3WewSQoRvD9TF01ss5ZJfG2Hhdc5TPLmU/8BJKiP4DUEsDBBQAAAAIANqcWF0qBlL5vg0AAP0jAAAdAAAAYWRhcHRlci9yZWNvdmVyX2FkZGl0aW9uYWwucHm9Wt9z47YRfvdfwbs+gDxTVC7TdjJW+KDY9FmNLbmSfEnqejgQCUmIKZIHkLZ0af737uKHSFl07trO1A82CYCLxWL3228Bv337NtpWTOQ0c5aCyXWvFEXCpHQES4onJnbOshBOtWbQXXxmuUPTlFe8wA9STld5ISueOLLmFQvevn17wjdlISqHilVJhWT2PSnyim2rjC9syzLJq8y+rKlct/r0n1bDr7LIT0CDjVPSCkeaIc4tvNpBciftYyVowhY0ebQNn3m55BkzMvQaY1jjLsl4vrLSlAni50I8MiFPTq6iaRTiDG4c49dx7AUwosiemOsFsD6WVyc3k4u762gWuiSvN+WO+CRnlfxU8xQey126yymFp08Fzaju/ETlhngnJylbOnJN3dI7O3HgZx0aMwTQ+u1f/up6qvmZV2u1ThgYFCXLXSIWxHOodJb6Q/zBbQKr5A5shHAzulmk9GwJ6tLUff/Nt39+h788f0GId7YO6jKlFXMXegbBqlrkzjpYs23KV0xWrlFPfQ76mRG4DUFW0FS6ViEcEePWup6nPnkWoIFb+luzqjI0QwelMVmweUy5cPWLDOeiZj7bclnFxaN68wbVpgzLAFce53QD4gL8c0oC6CBaaXgK1Fx6dqVaCjsg3a0vYTPjR7Yzwnmewkzht94p+WdOlHTQu8zAR0AvvdIN2xRiF7P8iYsi38B41yygqKvwt99PrJHRAdHO96SPftSXLFv2k5Uo6pL476AV3LC/lKapT0636rMtfuMSPU2woVvwBfOS1AIt0TSUjD42b+wJzUS8h4dmt/nSeAT88gIulXtahe2Pcpwm8AJZlyW4r3Qns0iIQnhnsLR7FPEQtqS1djSQleCl8UMcTGQCHkgeQjJ8ojyji4yB5SoK3kQHzpozQUWy3vVEUUDMKws4TzSrmXQ2dOfk0LpgCk4kKAIw4mR8w6uAtP0QJtJ7Ugi+4oA08aZI64xJUDB5pCtm1gmGDlDlgOeSicr9xgd17Zg+kUUtEkY84+PhHlQC/WSkukRWdbqLRZ3nTIB3pF8amTLJV7lxQ7sTIujAiDeh0aZpsnr127MGgBtngnLJnI9oLbU9LrnL2bZkScVSpz3aoBU5CF7hp9poANp8uYv1LNIVxlhWUqhCGpGtT6bR+eRjNP0lnk3upudRfDWcXUWzAEPJCKdJVdMs/E3H35lGKx0G6M8YZ0xLC1ZZsXDJO1yK5/1ubWOnfRNqUR2rnNpMo1V2kjXNVyxt7CsCxEVYC9hTaS+C6WQy75MutTsmuDzKXJ0zaTwKRXA7nEbj+T7gIUEwFbx6AEx7NQR0nt3dzIJqC9twGDFlxiv8RLbDUaOqj0YMsVMPc4njEP+91w5rZWI9EY6GJetvX1+XHu0YT7OLOnPIqRLQdhK9B9pPEpplsY0wV/iQsZ8s4mVpiMGFTRprxa5Zi+0J7zHcWn7vv8Pmh/1AhT8HKdXdZxIBEMhzgyxLVCHbne0lgwJGyWKDy4vlpnhEZ/ZTf0k3PNv5MMRXso3Of3J+KGA6uyCHwq/dZ5jTUUi/cxhN1o6kTxBMgj47f5tNxteK5YgUAAmQarVGYDLC6tzYUVMJh9bgPIUInHHh5LTiT8xhT0VWo0M5XDqYYiA1Y57JwJP5ps4gwaaBtn5gtHFRa62PSXRNl1rMQaf2SlFURVJk4JfK90FCn9jGdqiC79jmN3awknk0vMOVZmjf/ffOhssNrZK1EZ0UNQTGN/uI4BgOAs3j2k/uyZg8tByeggrJmiWPZcEhleK6uTdYHLbqJfMD/6dozXEBIacJjX7pUPmGQwIB6qZcw2mkkkbctm00ek+UNyUVKDrYHZpo0e5sq7N9E+46Q6/NlY0KWgDE+nKJfkeBkVYwxwYiESOFe41kZdHTMAMytz2I0JQnYBfl4qHxdFnRqpYhuR3OZnH08/B8Hp9Pbm7x781wfn4VD8cX8XT4UzyfDgENp9Ht9fAX4msvjdmWJcpLZagm9fc6gFFiu32xZpwhok+Xh+lQ6xze7WPNNIoxhJDLbEDxvKwr2dcYPGgi9iDQIH5TSB5oT2tXuhKMITHDIITXvB10NuQwHIslEA0ud86nmuZVvWkiNbC0O6MY4C61HFVjZ0iDo3w9gKc6A5pKA/PU9GkWwjKdWF2S0FrSDJibKBa1rCAHAG1TTh2YLQ0JAB1xWAb+5NpWXwvKipUM7Rw21zRJq6fpYM8WZr2n96RvRQzw4y/zaq2w9iflada3ZvPhdB5dEI2unMnQLssH7oOGwT++gXF4NU8tb7JcTTMVvbtInPiSg3EuKSzZ3w/aO9KLAXtxe7cNyURhgXESE1ElsCDleI4s6XMOXmViceAswMtTaABcZwL+bijw8qZMAtZaY0LRXvmTWoZzo0yrYtspFkAmIUsMjPvBhhWJcqw+xChYFl0Md5pBkoBZYWhgMEeXPrgVfbTo/M6QEhPCrXLODLIEDGqq5JHY6o7q4g7bWvUdlsvBEhtd/OXrhuvJ+Y8ACv9qvY1/aHDmIHErmPHT8FVaPQAeX0L5GB5xyMGCAfCzsKFiB1IV+kKndj3YXbbM+GpdmWVqKOrtm41VMKLK4lASBIsSBpWGshl5eKORz/LjKWAY31gcHutUvJfsLNA2mv04ZOCQ4FfICa4RyfAjEOkdzokpTYeR4rTG8Q/tZsdhGOhKToE+xPqyhoj2jkcbiMUCxUZ13yD7KemRU5TkIW4eGKVTDF+qyslIC1RMI5fC2o7nNXtt7tAmP1XYvSYa+oMVq1xrcW9v8T+Uj9ZAgonWMGcg3UZopoEkixsBO6AmNPzWbM5xV0feNfutHddR0ZfTPHmF9dqfdlxGP0fnd/PRZBzfTiH2xsPxeWT8USGiCQGb4cyrr0+/LKyZTh0U/tGqy121BuRCOqtxDOtkX7fGEFtY+Kpe89yBqxrsw65TieP59OJDsj/K07PCBimwkfUCauw+AKGDRwB40FXkkDMaVNWz+i0W5TdkEDOxIqkSyFiC/g9RDrv+BP6fNkQ5IN5RKOu0AVlPCSZn2sOa2DakZjycjz5GsCHR5fXow9U8Bmr+CxloSnQIXwAY4T3p9fCBqHrfoMiDng+fdXq9fzj48olmPKWaCL3o+tr41xH1MpI1DhxH18v66p5A9Q540evpT4mtZKAFrQRdFlJgCMzFhF4gPHrYZlzDrtq8QtWFa344VkDlx/AAgRsjkP/nCnSd9t+sobVrAS2R67lfUxIeCzpIzTeTH6M4+vvd6OPwOmrivzWZNygFViitU8VW733v/QNQ4qwG+za8qsvvtT1e8/vLaTS7Qhw6j+BNq/Wq6391loJ81OGmOk0dQ9XLZSoQVEeOIUEpG2a32dYjnSDfIAeeqjNeVhJpgEQ5WOVkkLJcF5Xok2YsJJv9uZGuG7wOeMMfyH8AcPN1iwqawzDFVYCx6gIfSvu+LRiQqjgao52UJTzFcvF5DXVZVQCSAINDzHp1H//LEFCGPooAaPyPA0B7iz2rPygDL0c/RxfxOL4YDT+MJ7P56HwWj8bzaDoeXsfDu4vRnPit6ijG+WOwTWzOF8htNL4YjT+0PIJtE1ZWzg9Uskg9Is8FGgrtZ1+h1uVwdA0qAYbPoimA+flVdP7j7QSUmhFfEa+42gHlxl8uyPSCWB3txzHkOimx+kKTYI+/v70J908B7OKGVlDOJu6L8FYsoeHJ5izpZUgS7QgEgwcJfYYEysblYXbXFDfeZzaI14b3hibrd5jk3pzYx0pPXsF3Xfl78MUyQZWobKuOngV74uzZNQFt3M3QTVua6gNo/dI6fR58TWnaVZj+z2WpFqB1LArQ8N5tEEmzX4g786CRrQ1qmml30eyHRqbNB8qQsC47Pzk1NvIGG5rzJZOVTfgVFUAuXy7GqmaumPYC4PEPlqn3JfjM7X2UolrmmjH4By8v8UZGz+iTZ+Lvu0a38QUQnSHU26rO+3x4iYeL87Eq4OrKSK31yJuxHMCeL9QCj3nxnNvLq/YkpRUdCI2/LUzuWyw+zi0G2kNzLXjUj3EGKKOCzIw9Uzqoo2J3fwFoLG+G3KtvHlqe60HEtfoM3ybHtO1oHe+61DYGK5uLMjxc1HcaAfiQxL1zXX276BNdjau/K0gQr5dBgmUh3iRm+qitKlxUxRuoY369haekr0//YNSX7JWhXSqJuhFzTiEPu7oLLOvmNiJUCld1JyriL3YVk6Anoou60osl/8x8U8Oo7VE1l7o8a+2BMpK+8fGBj1CeszSGAkYdjwidMI+XhAhy9jmwt8CHxVi7ZnTJ+WQ8n06ur6OpsrRPovGH0TiKppCY4nk0m8/0DhyuutSUFsW8jIqyCYdGg2NkUHtyqJgZjiYg02g+BDX0merNcDy6BFUMQreIkrV6c7t8cJCrI9/cLaurDnsdbv81IhiKVY0p4RbfhIsX5AA4MTXNLh5Jwywp8ZN1wRPYRNfQD50bbMH18rNezxzoEF+lXEwPsIOfai4gF+h7h+NvDAq+/s0rc1ki1OhIFQHqzDg+mIPCNDrddKmhWVQjTFeO/os80Mhp8dpjYXu2pdaEZa397rtuE0BR6Q1oqIBKMuy0R10GQ95/3xTp34ffdV3awvv7/Y3Sd6a2lurIkuFBTy9BQmruv5vLGxqY7YZcbPb3TLPzQyawP4f27eGv57Xrhf3R9skJSLUkC4TGMfphHJMzc/X2b1BLAwQUAAAACADynFhd8GsHxgEHAAD+EgAAGAAAAGFkYXB0ZXIvdGVzdF9yZWNvdmVyeS5weZ1YbW/bNhD+7l8hBChIFaqSdGgxxNCHLHW7oF0SONkwwDAIWjrZnCVRI6nEbrH/viNF+d1BUn+wJYp3unvu7rmjT05OfpfavKuVTEHrIBcL0yjQ/aCSQcWNeITg34ZXpikDWEDaGCGrQFSBmYGGwIA2Oj45OemJspbKBFxNa640dPeprNJGKahMnDdOc8B1kObd8+l3UXfX/2hZdddS93Ily6DmZlaISeCX7/C222KgrHNRrF7VVMJYe1rB7i4uZTrvxFFbupL3LjMF6TItRDV1lrWX3R58Jh9BMZ5lwnrOC7vJry57vbvLq6+XXwaJtYtKHUP1KJSs4ikYSu7O2eWnT9cP17c3l9+Y30oibRR1+xmz5jMWxgiLLB6BhjGCh1jp0fn4lKSyxFuBqDDdCAOnVuHKEHZvmkyAJmG4oaDXyyAPRIZKhFnSRXihAGGvgkykhj7yooFkEdUiS9BatBKvaBg6oZyLAgPkZLjA4P4BpVTLgVJSUZIBhgEUNxBANRUVgLKQ+XwhXkV7x56kmoOimAw6vOgF+Ml5KYplJKJcFhmoxD7qqyhLOixjqQSqRb9KmTUFaOoBC/texGHWXodOp6oTovjT6Zuzj1lskyeefidvRD/FdYddanaftXLxk0I4va5TVUc/yFxUGbkgt58/f7u+GbDBzRf8GQyvb76wz9d/P/w5HJD/wv6OZFpHo3FrS5o4gFELLBIRoVkJ6vVWJLgRV5ie8fcfPiYqzsQUU3NtQNjtPLYlxS1tNbJVFerkPHriRcE0QlhlOonPzs4xu7hpdPKsJ+EhGEg6g3ReS4HpR04pWSGHsKF96HybSGmvlxYcqWLoI3dl5TRdFdwDfl1xDT7yNi/sOpMKXwMZK3nNhGYT2SBYGeNVxhoNmmVCG1EhBr4wMQU0FLlX40BGp5sSsmQ0Xq1Z9Vo2KsXkX+90KSdVICxVKV5NgX4ILzr5mNc1VBkVYX8poMgCsRJ8EmbWkUD8GctqdtdaM3CwYyWUfOHzWyfvQ0sHtZTF9qtRrilMYh/E6C7t6jHqLI3SWVPNtfgOyXm4bbVQ2iQVLAxttYR9i0KMiIMyA+TighZQ0c6XMHq/rUDJJ52MnJrobatijdaeqtFiRBwrkLEDbOEAQxXjqMB40A67MDyuw5rzA/Ugl+xq+S+MPiAnbeWBjX/pqAUstdhw13yKxLIX79cF4/xIMJyWDaOHltz0EKawoBsUF5GDvHbhYFhF0pNkNDobb8Vw18kVm4E3FduMxh+b8LnBjgKLFGpbxrtOd5JJmsfe3Tt8e+dy73m36F82mM6h8AAMHZi5BbNDjnaEv5alZE3sB+J+relh66LO+qN4YPcDXlo0MHvxx5X+BvXswvHCHjEBzDtA3pxxjZMJXZtdJ1mMKWZkKgtKUt5oXpCI6FLO0bt+PSI3ZJz8so1rN1sgmdlBgKvlJ4FWoIdL6tLMRO0wEcvJP7hOs4h0LyFRy5Ss7bbI3NtbMdPaLo+ZD3khpjOzI/KDtCyODenu8v4eO8++ijZ2JNpuuQc2csRr+R2Ov+L5nrfLqetWbMJX5FesmmoPfT8M7DIYLiYjFa+TwreoSIS7pP5LON4S3cjRB9UAxfZIUTY8YOoWTt7uiAwHdlQb3l4N7u/Z4O/B1Z8Pt0Oc2ZDBGeQ57k0unXqsXF8pQ5fKATc2aQxkgZHYANDfgAe2rRdgF9u8x2FtG9Aj5uQROVBdJHqODF+J8n4reDHkkYX0eD9YF2HUluVRMmgTxNYBSj8iEeCbsSk0FkaGRWIEqvspQlht784ldc3aASqylxxHZ7P2oIa0bblZbK/1GsOzjdR5ITM8VzF971TSDV12fj1zHzdrFTimxn4P6Tb7Q0FczjOh6HYUuy1ulGOTpe2iEyLtTLOG8sjE/nwOIqM5LHaIA1+j6Mg+GYe7bLMBMsFM3KOcFw3ZB3S6aEV2ThEZzgn2xIbD5M+p3y8/1AaiNtg7/LFlI/rusObPHDskovkjTqJuOliN0H46wDNZPC3khJK37zbC8O7tKsZkV9mhmcq9IYzOD0yA7hHOIHjw45kPO1bbTwR+q6vfSHMjK6CHmOBsb8Yphdao2Id83c1YyqtKGlbwpkpn2PH5dH+2e2FfX23BL5EvWTtAa6qea/k/18Jf1Zd/+3Z79XXwCUnWDbG4NOpGpsDjgkDZUwZU6ZKM91u4dyzCZMMzXIcAcaaJ6hGZeztVedL9txLf8BI0HheB4tcc0U08XlE77+uObpBf2gVsAu4AnqxSG5GGhORNgdfdBP0rytcysSlwgB2OjNDDBo82JfgZegVdMClkOoeM2H8gfAzbhKB8W3nrrNfMbOJYRCCjr2wxIg8Y5iJOlSxJCMNzhqgYIxfrf4JwgaIhE6nxLIaHt97/UEsBAhQDFAAAAAgACZ1YXfSEv1FMAgAALQgAABwAAAAAAAAAAAAAAKSBAAAAAGFkYXB0ZXIvTE9DQUxfVkFMSURBVElPTi50eHRQSwECFAMUAAAACADynFhdemlyfmkJAAD7EwAAEgAAAAAAAAAAAAAApIGGAgAAYWRhcHRlci9SRUFETUUudHh0UEsBAhQDFAAAAAgACZ1YXWEFpsTFAAAAGgEAACMAAAAAAAAAAAAAAKSBHwwAAGFkYXB0ZXIvUkVDT1ZFUllfU09VUkNFX0hBU0hFUy5qc29uUEsBAhQDFAAAAAgAy5xYXWYa/I4uBQAAhgwAABwAAAAAAAAAAAAAAKSBJQ0AAGFkYXB0ZXIvcHJvY2Vzc19yZWN5Y2xpbmcucHlQSwECFAMUAAAACADanFhdKgZS+b4NAAD9IwAAHQAAAAAAAAAAAAAApIGNEgAAYWRhcHRlci9yZWNvdmVyX2FkZGl0aW9uYWwucHlQSwECFAMUAAAACADynFhd8GsHxgEHAAD+EgAAGAAAAAAAAAAAAAAApIGGIAAAYWRhcHRlci90ZXN0X3JlY292ZXJ5LnB5UEsFBgAAAAAGAAYAtgEAAL0nAAAAAA=='
data = base64.b64decode(PAYLOAD)
assert hashlib.sha256(data).hexdigest() == PAYLOAD_SHA256
RECOVERY_DIR = RUN_ROOT / "additional-memory-recovery-v1"
with zipfile.ZipFile(io.BytesIO(data)) as z:
    for item in z.infolist():
        rel = Path(item.filename)
        if rel.is_absolute() or ".." in rel.parts: raise RuntimeError("Unsafe recovery path")
        target = RECOVERY_DIR / rel;body = z.read(item)
        if target.exists() and target.read_bytes() != body:
            raise RuntimeError("Existing recovery adapter differs; preserve it before using a newly versioned adapter.")
        target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists(): target.write_bytes(body)
ADAPTER = RECOVERY_DIR / "adapter"
print("Verified recovery adapter:", ADAPTER)


In [ ]:
previous = RUN_ROOT / "ENVIRONMENT_PROBE.json"
prior = []
if previous.exists():
    saved = json.loads(previous.read_text())
    if isinstance(saved,list):
        prior = [Path(x["python"]) for x in saved if "python" in x and not x.get("errors")]
candidates = ([Path(PYTHON_OVERRIDE)] if PYTHON_OVERRIDE else
              prior + [PROJECT_DIR / ".venv-p1/bin/python", Path(sys.executable), PROJECT_DIR / ".venv/bin/python"])
probe = """import importlib,json,sys
errors={}
for name in ('numpy','netsquid','pydynaa','qoala','netqasm'):
    try: importlib.import_module(name)
    except Exception as e: errors[name]=repr(e)
print(json.dumps({'python':sys.executable,'errors':errors}))
"""
PYTHON = None;attempts = []
for candidate in dict.fromkeys(candidates):
    if not candidate.is_file():continue
    result = subprocess.run([str(candidate),"-c",probe],capture_output=True,text=True,timeout=60)
    try: item=json.loads(result.stdout.strip().splitlines()[-1])
    except Exception:item={"python":str(candidate),"errors":{"probe":result.stderr or result.stdout}}
    attempts.append(item)
    if result.returncode==0 and not item["errors"]:
        PYTHON=str(candidate);break
(LOGDIR / "PYTHON_PROBES.json").write_text(json.dumps(attempts,indent=2)+"\n")
if PYTHON is None:raise RuntimeError("Existing native P1 Python unavailable. Select that kernel or set PYTHON_OVERRIDE. Nothing has resumed.")
print("Using native Python:",PYTHON)


Keep `FAMILY="all"` to recover causal then robustness in one allocation. For independent sessions with eight cores each, select `"causal"` in one session and `"robustness"` in another. Do not run `"all"` alongside a subset or restart the original additional launch cell against those same folders.

The first timing/application recovery uses a different run root and can run in its own allocation. CPU allocations do not guarantee enough RAM: if eight fresh streams still exceed the session allowance, increase allocated RAM or lower `WORKERS` and resume the same frozen run. Do not lower N or remove completed streams.


In [ ]:
from IPython.display import FileLink,display

def run_stream(command,log,env=None):
    command=list(map(str,command))
    with Path(log).open("a",buffering=1) as out:
        out.write("\nCOMMAND "+json.dumps(command)+"\n")
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True,env=env)
        try:
            for line in process.stdout:
                print(line,end="",flush=True);out.write(line)
            return process.wait()
        except BaseException:
            try:os.killpg(process.pid,signal.SIGTERM)
            except ProcessLookupError:pass
            try:process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                try:os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError:pass
            raise

test_env=dict(os.environ,P1_ADDITIONAL_PACKAGE=str(PACKAGE))
returncode=run_stream([PYTHON,ADAPTER / "test_recovery.py"],LOGDIR / "ENGINEERING_TESTS.log",env=test_env)
if returncode:raise RuntimeError("Recovery engineering checks failed; inspect their printed output.")
command=[PYTHON,ADAPTER / "recover_additional.py","run","--package",PACKAGE,"--results",RESULTS,
         "--family",FAMILY,"--mode",MODE,"--workers",WORKERS]
if QOALA_REPO:command += ["--repo",QOALA_REPO]
exit_code=None
try:
    exit_code=run_stream(command,LOGDIR / "CONTROLLER.log")
finally:
    export=[PYTHON,ADAPTER / "recover_additional.py","export","--package",PACKAGE,"--results",RESULTS,"--family",FAMILY]
    exported=subprocess.run(list(map(str,export)),capture_output=True,text=True)
    print(exported.stdout,exported.stderr)
    status=LOGDIR / "STATUS.json"
    if status.exists():print(status.read_text())
    log=LOGDIR / "CONTROLLER.log"
    if log.exists():display(FileLink(os.path.relpath(log,Path.cwd())))
    review=RUN_ROOT / (RESULTS.name+"-"+FAMILY+"-additional-memory-recovery-review.zip")
    if review.exists():display(FileLink(os.path.relpath(review,Path.cwd())))
if exit_code:
    raise RuntimeError("Additional recovery stopped. The exact status/exception and log link appear above. Keep all existing checkpoints.")


The controller reports how many checkpoint receipt files are present, then the unchanged original runner verifies their hashes before reuse. Stream numbers printed before the old crash do not necessarily count every finished checkpoint.

After full completion, retain the native raw data on the server and return the compact review ZIP. Independent full saved-raw replay is still required before updating paper claims. These studies remain diagnostic; recovery does not change their statistical precision or scope.

A fresh-worker memory failure, a smoke equivalence mismatch, and a parent-analysis allocation failure require different follow-up. Send the final status and controller log if recovery stops.
